# Data Preparation for the Physformer-CycleGAN


From the repositiory structure, it looks like we have to place the pngs or jpegs in trainA and trainB
So we if break it down, we need real world and physcial dataset train and test split that can be reproduce and then can be automatically placed in the corresponding folders. 


In [1]:
import os
import numpy as np
import json
import pickle
import matplotlib.pyplot as plt
import pandas as pd 
from PIL import Image
np.random.seed(42)

# Saving the Source Data properly into the CycleGan Repo

In [2]:
def preprocess_image(img):
    img = img.astype(np.float32)
    img = (img - np.min(img)) / (np.max(img) - np.min(img))
    # intesnity correction and multiplying it with the 255
    # img = img**1.5 
    img = img*255
    img = img.astype(np.uint8)
    return img

# At first, we locaate all the npz files of the physcial dataset and then decode them into pngs
# then we need to split the data into train and test set
# then we copy the BUSI train and test images similarly in the folder. 
# Then we move this folder outside to the data folder to avoid the memory issues. 

# new physcial datase path for npz files are where
# lets think about the train and test split for the 
# Lets read the npz files 
source_npz_files = np.load("/home/user/haris/data/ultrasound_scans_dataset_iv/training_data/dataset_iv/data_npz/dataset_iv_split.npz")
source_train_arrays = source_npz_files['X_train']
source_val_arrays = source_npz_files['X_val']
source_test_arrays = source_npz_files['X_test']

# load the json for proper indexing  of the images
json_path = "/home/user/haris/data/ultrasound_scans_dataset_iv/training_data/dataset_iv/data_npz/dataset_iv_split_info.json"
with open(json_path,"r") as f:
    split_info = json.load(f)

train_indices = split_info['train_indices']
val_indices = split_info['val_indices']
test_indices = split_info['test_indices']



dataset_name = "synt2realUS"
images_save_dir = f"/home/user/haris/pytorch-CycleGAN-and-pix2pix/datasets/{dataset_name}"

source_train_images_save_path = os.path.join(images_save_dir,"trainA")
source_test_images_save_path = os.path.join(images_save_dir,"testA")
os.makedirs(source_train_images_save_path,exist_ok=True)
os.makedirs(source_test_images_save_path,exist_ok=True)

# print(source_test_images_save_path)

for idx, img in enumerate(source_train_arrays):
    img = preprocess_image(img)
    img = Image.fromarray(img)
    img_id = train_indices[idx]
    # lets write the image without any axis, labels, ticks for the cyclegan to use
    img.save(os.path.join(source_train_images_save_path,f"img_{img_id}.jpg"))
    # break

for idx, img in enumerate(source_val_arrays):
    img = preprocess_image(img)
    img = Image.fromarray(img)
    img_id = val_indices[idx]
    img.save(os.path.join(source_train_images_save_path,f"img_{img_id}.jpg"))
    # break

for idx, img in enumerate(source_test_arrays):
    img = preprocess_image(img)
    img = Image.fromarray(img)
    img_id = test_indices[idx]
    # lets write the image without any axis, labels, ticks for the cyclegan to use
    img.save(os.path.join(source_test_images_save_path,f"img_{img_id}.jpg"))
    # break

# Lets see the images in the trainA folder
images_train_source = os.listdir("/home/user/haris/pytorch-CycleGAN-and-pix2pix/datasets/synt2realUS/trainA")
images_test_source = os.listdir("/home/user/haris/pytorch-CycleGAN-and-pix2pix/datasets/synt2realUS/testA")
# images_train_source = [int(img.split(".")[0].split("_")[1]) for img in images_train_source]
print("Number of images in the trainA folder", len(images_train_source))
print("Number of images in the testA folder", len(images_test_source))

# Lets get the training and testing images from the BUSI folder and save their images into the folder


In [3]:
dataset_name = "synt2realUS"
images_save_dir = f"/home/user/haris/pytorch-CycleGAN-and-pix2pix/datasets/{dataset_name}"

# BUSI dataset path 
benign_images_path = "/home/user/haris/data/data_busi/Dataset_BUSI_with_GT/benign"
malignant_images_path = "/home/user/haris/data/data_busi/Dataset_BUSI_with_GT/malignant"

benign_images = os.listdir(benign_images_path)
malignant_images = os.listdir(malignant_images_path)

# all_busi_images = benign_images + malignant_images
all_busi_images_paths = [os.path.join(benign_images_path,img) for img in benign_images if not "mask" in img] + [os.path.join(malignant_images_path,img) for img in malignant_images if not "mask" in img]

target_train_images_save_path = os.path.join(images_save_dir,"trainB")
target_test_images_save_path = os.path.join(images_save_dir,"testB")
os.makedirs(target_train_images_save_path,exist_ok=True)
os.makedirs(target_test_images_save_path,exist_ok=True)


In [4]:
np.random.seed(42)
df = pd.DataFrame({"image_path":all_busi_images_paths})
df["image_key"] = [x.split(".")[0].split("/")[-1] for x in df["image_path"] if not "mask" in x]
df["image_type"] = ["benign" if "benign" in x else "malignant" for x in df["image_key"]]
df["image_type"] = df["image_type"].astype(str)
df["image_type"] = df["image_type"].replace({"benign":"0","malignant":"1"})
df["image_type"] = df["image_type"].astype(int)
df.drop_duplicates(subset="image_key",inplace=True,keep="first")
df.sort_values(by="image_key",inplace=True)
# df.to_csv("busi_images_info.csv",index=False)
# Randomly pick the train and test split test
train_indices = np.random.choice(df.index,size=int(len(df)*0.8),replace=False)
test_indices = np.setdiff1d(df.index,train_indices)
df.loc[train_indices,"split"] = "train"
df.loc[test_indices,"split"] = "test"
df.to_csv("datasets/busi_images_split_info.csv",index=False)


In [5]:
import shutil
# Lets call each image path and save it in the trainB and testB folder
for idx,row in df.iterrows():
    if row["split"] == "train":
        shutil.copy(row["image_path"],os.path.join(target_train_images_save_path,f"img_{row['image_key']}.jpg"))
    else:
        shutil.copy(row["image_path"],os.path.join(target_test_images_save_path,f"img_{row['image_key']}.jpg"))
    # break


In [6]:
# prin the length of the train and test images
print(len(os.listdir(target_train_images_save_path)))
print(len(os.listdir(target_test_images_save_path)))


517
130


In [12]:
import cv2

In [14]:
cv2.imread(os.path.join(target_train_images_save_path,os.listdir(target_train_images_save_path)[0])).shape

(463, 557, 3)